Данный код реализует серверную часть проекта.
1. OCR: обработка PDF сгенерированной амбулаторной карты пациента и преобразование файла в машиночитаемый текст.
2. NER: выделение именнованных сущностей, а именно назначений в формате - наименование назанчения, время, дата, кабинет.
3. Создание JSON и шифрование с помощью AES-256, загрузка в Яндекс.Диск

# OCR

**Программно генерируемые PDF**: эти PDF создаются на компьютере или при помощи технологий W3C (HTML, CSS и Javascript), или при помощи другого ПО, например, Adobe Acrobat. Такой тип файла может содержать различные компоненты, например, изображения, текст и ссылки, по всем ним можно выполнять поиск и легко их редактировать.

Отсканированные документы с OCR **Текст, выделенный полужирным шрифтом**: в этом случае после сканирования документа применяется ПО для оптического распознавания символов (Optical Character Recognition, OCR), распознающее текст в каждом изображении файла и преобразующее его в текст с возможностью поиска по нему и редактирования. Затем ПО добавляет на изображение слой текста, благодаря чему при просмотре файла можно выбирать его как отдельный компонент.

In [58]:
pip install PyPDF2

In [59]:
pip install pdfminer.six

In [60]:
pip install pdfplumber

In [61]:
pip install pdf2image

In [62]:
pip install Pillow

In [63]:
# Для считывания PDF
import PyPDF2
# Для анализа структуры PDF и извлечения текста
from pdfminer.high_level import extract_pages, extract_text
from pdfminer.layout import LTTextContainer, LTChar, LTRect, LTFigure
# Для извлечения текста из таблиц в PDF
import pdfplumber
# Для извлечения изображений из PDF
from PIL import Image
from pdf2image import convert_from_path
# Для удаления дополнительно созданных файлов
import os

In [64]:
# Создаём функцию для извлечения текста

def text_extraction(element):
    # Извлекаем текст из вложенного текстового элемента
    line_text = element.get_text()

    # Находим форматы текста
    # Инициализируем список со всеми форматами, встречающимися в строке текста
    line_formats = []
    for text_line in element:
        if isinstance(text_line, LTTextContainer):
            # Итеративно обходим каждый символ в строке текста
            for character in text_line:
                if isinstance(character, LTChar):
                    # Добавляем к символу название шрифта
                    line_formats.append(character.fontname)
                    # Добавляем к символу размер шрифта
                    line_formats.append(character.size)
    # Находим уникальные размеры и названия шрифтов в строке
    format_per_line = list(set(line_formats))

    # Возвращаем кортеж с текстом в каждой строке вместе с его форматом
    return (line_text, format_per_line)

In [65]:
# Создаём функцию для вырезания элементов изображений из PDF
def crop_image(element, pageObj):
    # Получаем координаты для вырезания изображения из PDF
    [image_left, image_top, image_right, image_bottom] = [element.x0,element.y0,element.x1,element.y1]
    # Обрезаем страницу по координатам (left, bottom, right, top)
    pageObj.mediabox.lower_left = (image_left, image_bottom)
    pageObj.mediabox.upper_right = (image_right, image_top)
    # Сохраняем обрезанную страницу в новый PDF
    cropped_pdf_writer = PyPDF2.PdfWriter()
    cropped_pdf_writer.add_page(pageObj)
    # Сохраняем обрезанный PDF в новый файл
    with open('cropped_image.pdf', 'wb') as cropped_pdf_file:
        cropped_pdf_writer.write(cropped_pdf_file)

# Создаём функцию для преобразования PDF в изображения
def convert_to_images(input_file,):
    images = convert_from_path(input_file)
    image = images[0]
    output_file = "PDF_image.png"
    image.save(output_file, "PNG")

# Создаём функцию для считывания текста из изображений
def image_to_text(image_path):
    # Считываем изображение
    img = Image.open(image_path)
    # Извлекаем текст из изображения
    text = pytesseract.image_to_string(img)
    return text

In [66]:
# Извлечение таблиц из страницы

def extract_table(pdf_path, page_num, table_num):
    # Открываем файл pdf
    pdf = pdfplumber.open(pdf_path)
    # Находим исследуемую страницу
    table_page = pdf.pages[page_num]
    # Извлекаем соответствующую таблицу
    table = table_page.extract_tables()[table_num]
    return table

# Преобразуем таблицу в соответствующий формат
def table_converter(table):
    table_string = ''
    # Итеративно обходим каждую строку в таблице
    for row_num in range(len(table)):
        row = table[row_num]
        # Удаляем разрыв строки из текста с переносом
        cleaned_row = [item.replace('\n', ' ') if item is not None and '\n' in item else 'None' if item is None else item for item in row]
        # Преобразуем таблицу в строку
        table_string+=('|'+'|'.join(cleaned_row)+'|'+'\n')
    # Удаляем последний разрыв строки
    table_string = table_string[:-1]
    return table_string

Рассмотрим пример обработки на двух амбулаторных карт пациентов. Назначения пациентам записываются в таблицу, пример которой представлен на 3 странице, поэтому будем извлекать только ее.

In [67]:
# Находим путь к PDF
pdf_path = '/content/амбулаторная карта 1.pdf'

# создаём объект файла PDF
pdfFileObj = open(pdf_path, 'rb')
# создаём объект считывателя PDF
pdfReaded = PyPDF2.PdfReader(pdfFileObj)

# Создаём словарь для извлечения текста из каждого изображения
text_per_page = {}
# Извлекаем страницы из PDF
for pagenum, page in enumerate(extract_pages(pdf_path)):

    # Инициализируем переменные, необходимые для извлечения текста со страницы
    pageObj = pdfReaded.pages[pagenum]
    page_text = []
    line_format = []
    text_from_images = []
    text_from_tables = []
    page_content = []
    # Инициализируем количество исследованных таблиц
    table_num = 0
    first_element= True
    table_extraction_flag= False
    # Открываем файл pdf
    pdf = pdfplumber.open(pdf_path)
    # Находим исследуемую страницу
    page_tables = pdf.pages[pagenum]
    # Находим количество таблиц на странице
    tables = page_tables.find_tables()


    # Находим все элементы
    page_elements = [(element.y1, element) for element in page._objs]
    # Сортируем все элементы по порядку нахождения на странице
    page_elements.sort(key=lambda a: a[0], reverse=True)

    # Находим элементы, составляющие страницу
    for i, component in enumerate(page_elements):
        # Извлекаем положение верхнего края элемента в PDF
        pos = component[0]
        # Извлекаем элемент структуры страницы
        element = component[1]

        # Проверяем, является ли элемент текстовым
        if isinstance(element, LTTextContainer):
            # Проверяем, находится ли текст в таблице
            if table_extraction_flag == False:
                # Используем функцию извлечения текста и формата для каждого текстового элемента
                (line_text, format_per_line) = text_extraction(element)
                # Добавляем текст каждой строки к тексту страницы
                page_text.append(line_text)
                # Добавляем формат каждой строки, содержащей текст
                line_format.append(format_per_line)
                page_content.append(line_text)
            else:
                # Пропускаем текст, находящийся в таблице
                pass

        # Проверяем элементы на наличие изображений
        if isinstance(element, LTFigure):
            # Вырезаем изображение из PDF
            crop_image(element, pageObj)
            # Преобразуем обрезанный pdf в изображение
            convert_to_images('cropped_image.pdf')
            # Извлекаем текст из изображения
            image_text = image_to_text('PDF_image.png')
            text_from_images.append(image_text)
            page_content.append(image_text)
            # Добавляем условное обозначение в списки текста и формата
            page_text.append('image')
            line_format.append('image')

        # Проверяем элементы на наличие таблиц
        if isinstance(element, LTRect):
            # Если первый прямоугольный элемент
            if first_element == True and (table_num + 1) <= len(tables):
                # Находим ограничивающий прямоугольник таблицы
                lower_side = page.bbox[3] - tables[table_num].bbox[3]
                upper_side = element.y1
                # Извлекаем информацию из таблицы
                table = extract_table(pdf_path, pagenum, table_num)
                # Преобразуем информацию таблицы в формат структурированной строки
                table_string = table_converter(table)
                # Добавляем строку таблицы в список
                text_from_tables.append(table_string)
                page_content.append(table_string)
                # Устанавливаем флаг True, чтобы избежать повторения содержимого
                table_extraction_flag = True
                # Преобразуем в другой элемент
                first_element = False
                # Добавляем условное обозначение в списки текста и формата
                page_text.append('table')
                line_format.append('table')

            # Проверяем, извлекли ли мы уже таблицы из этой страницы
            if element.y0 >= lower_side and element.y1 <= upper_side:
                pass
            elif i + 1 < len(page_elements) and not isinstance(page_elements[i + 1][1], LTRect):
                table_extraction_flag = False
                first_element = True
                table_num += 1


    # Создаём ключ для словаря
    dctkey = 'Page_'+str(pagenum)
    # Добавляем список списков как значение ключа страницы
    text_per_page[dctkey]= [page_text, line_format, text_from_images,text_from_tables, page_content]

# Закрываем объект файла pdf
pdfFileObj.close()

# Удаляем созданные дополнительные файлы
# os.remove('cropped_image.pdf')
# os.remove('PDF_image.png')

# Удаляем содержимое страницы
result1 = ''.join(text_per_page['Page_2'][4])
print(result1)

стр. 3 ф. № 025/у 
24. Записи врачей-специалистов: 
Дата осмотра 
Врач (специальность)  Терапевт 
Жалобы пациента  Отдышка, головокружение, слабость, кашель 
10.05.26 
на приеме, на дому, в фельдшерско-акушерском пункте, прочее. 
|Назначения (исследования, консультации)|Лекарственные препараты, физиотерапия|
|Повторный прием терапевта – 10.06.26 в 13 в 404 каб||
|Анализ крови – 5.06.26 в 7:30 в 12 кабинете||
|Флюорография – 6.06.26 в 9 в 101 каб||
|Листок нетрудоспособности, справка|Льготные рецепты|
|||
|Информированное добровольное согласие на медицинское вмешательство, отказ от медицинского вмешательства|None|
|Врач|None| 



In [68]:
# Находим путь к PDF
pdf_path = '/content/амбулаторная карта 2.pdf'

# создаём объект файла PDF
pdfFileObj = open(pdf_path, 'rb')
# создаём объект считывателя PDF
pdfReaded = PyPDF2.PdfReader(pdfFileObj)

# Создаём словарь для извлечения текста из каждого изображения
text_per_page = {}
# Извлекаем страницы из PDF
for pagenum, page in enumerate(extract_pages(pdf_path)):

    # Инициализируем переменные, необходимые для извлечения текста со страницы
    pageObj = pdfReaded.pages[pagenum]
    page_text = []
    line_format = []
    text_from_images = []
    text_from_tables = []
    page_content = []
    # Инициализируем количество исследованных таблиц
    table_num = 0
    first_element= True
    table_extraction_flag= False
    # Открываем файл pdf
    pdf = pdfplumber.open(pdf_path)
    # Находим исследуемую страницу
    page_tables = pdf.pages[pagenum]
    # Находим количество таблиц на странице
    tables = page_tables.find_tables()


    # Находим все элементы
    page_elements = [(element.y1, element) for element in page._objs]
    # Сортируем все элементы по порядку нахождения на странице
    page_elements.sort(key=lambda a: a[0], reverse=True)

    # Находим элементы, составляющие страницу
    for i, component in enumerate(page_elements):
        # Извлекаем положение верхнего края элемента в PDF
        pos = component[0]
        # Извлекаем элемент структуры страницы
        element = component[1]

        # Проверяем, является ли элемент текстовым
        if isinstance(element, LTTextContainer):
            # Проверяем, находится ли текст в таблице
            if table_extraction_flag == False:
                # Используем функцию извлечения текста и формата для каждого текстового элемента
                (line_text, format_per_line) = text_extraction(element)
                # Добавляем текст каждой строки к тексту страницы
                page_text.append(line_text)
                # Добавляем формат каждой строки, содержащей текст
                line_format.append(format_per_line)
                page_content.append(line_text)
            else:
                # Пропускаем текст, находящийся в таблице
                pass

        # Проверяем элементы на наличие изображений
        if isinstance(element, LTFigure):
            # Вырезаем изображение из PDF
            crop_image(element, pageObj)
            # Преобразуем обрезанный pdf в изображение
            convert_to_images('cropped_image.pdf')
            # Извлекаем текст из изображения
            image_text = image_to_text('PDF_image.png')
            text_from_images.append(image_text)
            page_content.append(image_text)
            # Добавляем условное обозначение в списки текста и формата
            page_text.append('image')
            line_format.append('image')

        # Проверяем элементы на наличие таблиц
        if isinstance(element, LTRect):
            # Если первый прямоугольный элемент
            if first_element == True and (table_num + 1) <= len(tables):
                # Находим ограничивающий прямоугольник таблицы
                lower_side = page.bbox[3] - tables[table_num].bbox[3]
                upper_side = element.y1
                # Извлекаем информацию из таблицы
                table = extract_table(pdf_path, pagenum, table_num)
                # Преобразуем информацию таблицы в формат структурированной строки
                table_string = table_converter(table)
                # Добавляем строку таблицы в список
                text_from_tables.append(table_string)
                page_content.append(table_string)
                # Устанавливаем флаг True, чтобы избежать повторения содержимого
                table_extraction_flag = True
                # Преобразуем в другой элемент
                first_element = False
                # Добавляем условное обозначение в списки текста и формата
                page_text.append('table')
                line_format.append('table')

            # Проверяем, извлекли ли мы уже таблицы из этой страницы
            if element.y0 >= lower_side and element.y1 <= upper_side:
                pass
            elif i + 1 < len(page_elements) and not isinstance(page_elements[i + 1][1], LTRect):
                table_extraction_flag = False
                first_element = True
                table_num += 1


    # Создаём ключ для словаря
    dctkey = 'Page_'+str(pagenum)
    # Добавляем список списков как значение ключа страницы
    text_per_page[dctkey]= [page_text, line_format, text_from_images,text_from_tables, page_content]

# Закрываем объект файла pdf
pdfFileObj.close()

# Удаляем созданные дополнительные файлы
# os.remove('cropped_image.pdf')
# os.remove('PDF_image.png')

# Удаляем содержимое страницы
result2 = ''.join(text_per_page['Page_2'][4])
print(result2)

стр. 3 ф. № 025/у 
 
24. Записи врачей-специалистов: 
03.03.2025 
Дата осмотра 
Врач (специальность) 
Жалобы пациента 
 на приеме, на дому, в фельдшерско-акушерском пункте, прочее. 
|Назначения (исследования, консультации)|Лекарственные препараты, физиотерапия|
|ЭКГ с динамическим мониторингом – 10.03.25 в 17:00 в 223 кб|Бета-блокаторы, ингибиторы АПФ, рекомендации по фи- зической реабилитации.|
|Стресс-тест – 03.03.25 в 12:00 в 103 кб||
|Консультация по коррекции терапии – 03.03.25 в 15:00 в 301 кб||
|Листок нетрудоспособности, справка|Льготные рецепты|
|Выдан на 7 календарных дней.|Оформлены.|
|Информированное добровольное согласие на медицинское вмешательство, отказ от медицинского вмешательства Получено|None|
|Врач д-р Иванов С.С.|None| 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 



# NER

In [69]:
import re

# Регулярное выражение для поиска назначений
appointment_pattern = re.compile(
    r"Назначения \(исследования, консультации\).*?\n(\|.*\|.*\n(?!.*Листок нетрудоспособности.*\n)(\|.*\|\n)*)",  # Ищем таблицу, останавливаемся на "Листок нетрудоспособности"
    re.DOTALL  # Для поиска по многострочному тексту
)

# Поиск таблиц с назначениями
tables = appointment_pattern.findall(result1)

# Обработка найденных таблиц
if tables:
    print("Найденные назначения:")
    for table in tables:
        # Извлекаем строки таблицы
        rows = table[0].strip().split("\n")
        for row in rows:
            # Убираем лишние пробелы и разделяем на ячейки
            cells = [cell.strip() for cell in row.split("|") if cell.strip()]
            # print(cells)
            # Проверяем, что строка содержит назначение (исключаем заголовок и лекарства)
            if "Листок нетрудоспособности" in cells[0]:
              break
            else:
                print(f"- {cells[0]}")
else:
    print("Таблицы с назначениями не найдены.")

Найденные назначения:
- Повторный прием терапевта – 10.06.26 в 13 в 404 каб
- Анализ крови – 5.06.26 в 7:30 в 12 кабинете
- Флюорография – 6.06.26 в 9 в 101 каб


In [70]:
import re

# Регулярное выражение для поиска назначений
appointment_pattern = re.compile(
    r"Назначения \(исследования, консультации\).*?\n(\|.*\|.*\n(?!.*Листок нетрудоспособности.*\n)(\|.*\|\n)*)",  # Ищем таблицу, останавливаемся на "Листок нетрудоспособности"
    re.DOTALL  # Для поиска по многострочному тексту
)

# Поиск таблиц с назначениями
tables = appointment_pattern.findall(result2)

# Обработка найденных таблиц
if tables:
    print("Найденные назначения:")
    for table in tables:
        # Извлекаем строки таблицы
        rows = table[0].strip().split("\n")
        for row in rows:
            # Убираем лишние пробелы и разделяем на ячейки
            cells = [cell.strip() for cell in row.split("|") if cell.strip()]
            # print(cells)
            # Проверяем, что строка содержит назначение (исключаем заголовок и лекарства)
            if "Листок нетрудоспособности" in cells[0]:
              break
            else:
                print(f"- {cells[0]}")
else:
    print("Таблицы с назначениями не найдены.")

Найденные назначения:
- ЭКГ с динамическим мониторингом – 10.03.25 в 17:00 в 223 кб
- Стресс-тест – 03.03.25 в 12:00 в 103 кб
- Консультация по коррекции терапии – 03.03.25 в 15:00 в 301 кб


# Работа с JSON

Формируем JSON с назначениями для двух пациентов.

In [71]:
import re
import json
import uuid

# Генерация уникального ID пациента
patient_id = str(uuid.uuid4())

# Регулярное выражение для поиска назначений
appointment_pattern = re.compile(
    r'Назначения \(исследования, консультации\).*?\n(\|.*\|.*\n(?!.*Листок нетрудоспособности.*\n)(\|.*\|\n)*)',
    re.DOTALL
)

# Поиск таблиц с назначениями
tables = appointment_pattern.findall(result1)

appointments_json = {"patient_id": patient_id, "appointments": []}

# Функция для форматирования даты
def format_date(date_str):
    """Преобразует 5.06.26 в 05.06.26"""
    day, month, year = date_str.split('.')
    return f"{int(day):02d}.{int(month):02d}.{year}"

# Регулярное выражение для извлечения даты, времени и кабинета
details_pattern = re.compile(
    r'–\s*(\d{1,2}\.\d{2}\.\d{2})\s+в\s+(\d{1,2})(?::(\d{2}))?\s+в\s+(\d{1,4})(?:\s*(?:каб|кабинете|кабинет|каб\.))?',
    re.IGNORECASE
)

# Обработка найденных таблиц
if tables:
    for table in tables:
        rows = table[0].strip().split("\n")
        for row in rows:
            cells = [cell.strip() for cell in row.split("|") if cell.strip()]
            print(f"Обрабатываем строку: {cells}")

            if not cells:
                continue

            if len(cells) > 0 and "Листок нетрудоспособности" in cells[0]:
                continue

            if len(cells) >= 1:
                description = cells[0]
                match = details_pattern.search(description)

                if match:
                    date = format_date(match.group(1))  # Форматируем дату
                    hour = match.group(2)
                    minute = match.group(3) if match.group(3) else "00"
                    cabinet = re.sub(r'(каб|кабинете|кабинет|каб\.)', '', match.group(4), flags=re.IGNORECASE).strip()
                    time = f"{hour}:{minute}"
                    description_clean = description.split('–')[0].strip()

                    appointments_json["appointments"].append({
                        "description": description_clean,
                        "date": date,
                        "time": time,
                        "cabinet": cabinet
                    })

                    print(f"  Добавлено: {description_clean} | {date} | {time} | {cabinet}")
                else:
                    print(f"  Не удалось извлечь данные из: {description}")

# Преобразуем в JSON
json_data1 = json.dumps(appointments_json, ensure_ascii=False, indent=4)

# Сохраняем в файл
with open('appointments1.json', 'w', encoding='utf-8') as f:
    f.write(json_data1)

print("\n" + "="*50)
print("JSON данные успешно сохранены!")
print(json_data1)

Обрабатываем строку: ['Повторный прием терапевта – 10.06.26 в 13 в 404 каб']
  Добавлено: Повторный прием терапевта | 10.06.26 | 13:00 | 404
Обрабатываем строку: ['Анализ крови – 5.06.26 в 7:30 в 12 кабинете']
  Добавлено: Анализ крови | 05.06.26 | 7:30 | 12
Обрабатываем строку: ['Флюорография – 6.06.26 в 9 в 101 каб']
  Добавлено: Флюорография | 06.06.26 | 9:00 | 101
Обрабатываем строку: ['Листок нетрудоспособности, справка', 'Льготные рецепты']
Обрабатываем строку: []
Обрабатываем строку: ['Информированное добровольное согласие на медицинское вмешательство, отказ от медицинского вмешательства', 'None']
  Не удалось извлечь данные из: Информированное добровольное согласие на медицинское вмешательство, отказ от медицинского вмешательства
Обрабатываем строку: ['Врач', 'None']
  Не удалось извлечь данные из: Врач

JSON данные успешно сохранены!
{
    "patient_id": "488abb8f-4e0b-4822-baa9-b4dfa01a6c7e",
    "appointments": [
        {
            "description": "Повторный прием терапевта

In [72]:
import re
import json
import uuid

# Генерация уникального ID пациента
patient_id = str(uuid.uuid4())

# Регулярное выражение для поиска назначений
appointment_pattern = re.compile(
    r'Назначения \(исследования, консультации\).*?\n(\|.*\|.*\n(?!.*Листок нетрудоспособности.*\n)(\|.*\|\n)*)',
    re.DOTALL
)

# Поиск таблиц с назначениями
tables = appointment_pattern.findall(result2)

appointments_json = {"patient_id": patient_id, "appointments": []}

# Функция для форматирования даты
def format_date(date_str):
    """Преобразует 5.06.26 в 05.06.26"""
    day, month, year = date_str.split('.')
    return f"{int(day):02d}.{int(month):02d}.{year}"

# Регулярное выражение для извлечения даты, времени и кабинета
details_pattern = re.compile(
    r'–\s*(\d{1,2}\.\d{2}\.\d{2})\s+в\s+(\d{1,2})(?::(\d{2}))?\s+в\s+(\d{1,4})(?:\s*(?:каб|кабинете|кабинет|каб\.))?',
    re.IGNORECASE
)

# Обработка найденных таблиц
if tables:
    for table in tables:
        rows = table[0].strip().split("\n")
        for row in rows:
            cells = [cell.strip() for cell in row.split("|") if cell.strip()]
            print(f"Обрабатываем строку: {cells}")

            if not cells:
                continue

            if len(cells) > 0 and "Листок нетрудоспособности" in cells[0]:
                continue

            if len(cells) >= 1:
                description = cells[0]
                match = details_pattern.search(description)

                if match:
                    date = format_date(match.group(1))  # Форматируем дату
                    hour = match.group(2)
                    minute = match.group(3) if match.group(3) else "00"
                    cabinet = re.sub(r'(каб|кабинете|кабинет|каб\.)', '', match.group(4), flags=re.IGNORECASE).strip()
                    time = f"{hour}:{minute}"
                    description_clean = description.split('–')[0].strip()

                    appointments_json["appointments"].append({
                        "description": description_clean,
                        "date": date,
                        "time": time,
                        "cabinet": cabinet
                    })

                    print(f"  Добавлено: {description_clean} | {date} | {time} | {cabinet}")
                else:
                    print(f"  Не удалось извлечь данные из: {description}")

# Преобразуем в JSON
json_data2 = json.dumps(appointments_json, ensure_ascii=False, indent=4)

# Сохраняем в файл
with open('appointments2.json', 'w', encoding='utf-8') as f:
    f.write(json_data2)

print("\n" + "="*50)
print("JSON данные успешно сохранены!")
print(json_data2)

Обрабатываем строку: ['ЭКГ с динамическим мониторингом – 10.03.25 в 17:00 в 223 кб', 'Бета-блокаторы, ингибиторы АПФ, рекомендации по фи- зической реабилитации.']
  Добавлено: ЭКГ с динамическим мониторингом | 10.03.25 | 17:00 | 223
Обрабатываем строку: ['Стресс-тест – 03.03.25 в 12:00 в 103 кб']
  Добавлено: Стресс-тест | 03.03.25 | 12:00 | 103
Обрабатываем строку: ['Консультация по коррекции терапии – 03.03.25 в 15:00 в 301 кб']
  Добавлено: Консультация по коррекции терапии | 03.03.25 | 15:00 | 301
Обрабатываем строку: ['Листок нетрудоспособности, справка', 'Льготные рецепты']
Обрабатываем строку: ['Выдан на 7 календарных дней.', 'Оформлены.']
  Не удалось извлечь данные из: Выдан на 7 календарных дней.
Обрабатываем строку: ['Информированное добровольное согласие на медицинское вмешательство, отказ от медицинского вмешательства Получено', 'None']
  Не удалось извлечь данные из: Информированное добровольное согласие на медицинское вмешательство, отказ от медицинского вмешательства По

In [73]:
!pip install pycryptodome

Каждый из полученных JSON-ов шифруем с помощью алгоритма AES-256.

In [74]:
import json
import base64
import os
import hashlib
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
from google.colab import userdata

# Секретный ключ из переменных окружения
FIXED_SECRET = userdata.get('ENCRYPT_SECRET')

def encrypt_json(json_filename: str, room_id: str, delete_original: bool = True) -> str:
    """
    Шифрует JSON файл (формат: IV + шифротекст в base64)

    Аргументы:
        json_filename: имя файла (например, 'appointments2.json')
        room_id: номер палаты (например, '28')
        delete_original: удалять исходный файл (True/False)

    Возвращает:
        имя зашифрованного файла
    """
    # Проверки
    if not FIXED_SECRET:
        raise ValueError("ENCRYPT_SECRET не найден в переменных окружения")
    if not os.path.exists(json_filename):
        raise FileNotFoundError(f"Файл {json_filename} не найден")

    # Формируем ключ из room_id + secret (как в примере)
    key = hashlib.sha256((room_id + FIXED_SECRET).encode()).digest()  # 32 байта

    # Генерируем случайный вектор инициализации
    iv = os.urandom(16)

    # Читаем исходный JSON как текст (как в примере)
    with open(json_filename, "r", encoding="utf-8") as f:
        plaintext = f.read().encode("utf-8")

    # Шифруем
    cipher = AES.new(key, AES.MODE_CBC, iv)
    ct = cipher.encrypt(pad(plaintext, AES.block_size))

    # Формируем результат: IV + шифротекст → base64 (как в примере)
    encrypted_data = base64.b64encode(iv + ct).decode()

    # Сохраняем зашифрованный файл
    output_file = f"room-{room_id}.json.enc"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(encrypted_data)

    # Удаляем исходный файл если нужно
    if delete_original:
        os.remove(json_filename)
        print(f"🗑️ Удалён исходный файл: {json_filename}")

    print(f"✅ {json_filename} -> {output_file}")
    print(f"   Размер: {len(encrypted_data)} символов base64")

    return output_file


# Функция для расшифровки (для проверки)
def decrypt_json(encrypted_file: str, room_id: str) -> dict:
    """
    Расшифровывает файл, зашифрованный функцией encrypt_json

    Аргументы:
        encrypted_file: имя зашифрованного файла
        room_id: номер палаты (должен совпадать с тем, что использовался при шифровании)

    Возвращает:
        расшифрованные данные (dict)
    """
    if not FIXED_SECRET:
        raise ValueError("ENCRYPT_SECRET не найден в переменных окружения")

    # Формируем ключ
    key = hashlib.sha256((room_id + FIXED_SECRET).encode()).digest()

    # Читаем зашифрованный файл
    with open(encrypted_file, 'r', encoding='utf-8') as f:
        encrypted_data = f.read()

    # Декодируем base64
    data = base64.b64decode(encrypted_data)

    # Извлекаем IV (первые 16 байт) и шифротекст
    iv = data[:16]
    ct = data[16:]

    # Расшифровываем
    from Crypto.Util.Padding import unpad
    cipher = AES.new(key, AES.MODE_CBC, iv)
    plaintext = unpad(cipher.decrypt(ct), AES.block_size)

    return json.loads(plaintext.decode('utf-8'))

In [75]:
file_name = input("Введите имя JSON файла: ")
room = input("Введите номер палаты: ")
encrypt_json(file_name, room)

Введите имя JSON файла: appointments1.json
Введите номер палаты: 27
🗑️ Удалён исходный файл: appointments1.json
✅ appointments1.json -> room-27.json.enc
   Размер: 832 символов base64


'room-27.json.enc'

In [76]:
file_name = input("Введите имя JSON файла: ")
room = input("Введите номер палаты: ")
encrypt_json(file_name, room)

Введите имя JSON файла: appointments2.json
Введите номер палаты: 28
🗑️ Удалён исходный файл: appointments2.json
✅ appointments2.json -> room-28.json.enc
   Размер: 920 символов base64


'room-28.json.enc'

Загружаем полученные зашифрованные JSON в Яндекс.Диск.

- appointments1.json описывает назначения для пациента в 27 палате (сентябрь 25 года);

- appointments2.json описывает назначения для пациента в 28 палате (июнь 26 года).

Этапы для получения токена, чтобы загружать файлы на Диск через код, описаны в [данной статье](https://ramziv.com/article/2).

In [77]:
02dimport json
import os
import webbrowser
import requests
from http.server import HTTPServer, BaseHTTPRequestHandler
import threading
import urllib.parse

# Константы
TOKEN_FILE = "yadisk_token.json"
CLIENT_ID = "b24baaede2184f0a8c4ab9033d2da2bc"
REDIRECT_URI = "https://oauth.yandex.ru/verification_code"
API_BASE = "https://cloud-api.yandex.net/v1/disk"

def get_token_automated():
    """
    Автоматизированное получение токена:
    1. Открывает браузер со страницей авторизации
    2. Пользователь авторизуется и получает токен
    3. Пользователь вставляет токен в консоль
    4. Токен сохраняется и возвращается
    """
    print("="*60)
    print("АВТОМАТИЗИРОВАННОЕ ПОЛУЧЕНИЕ ТОКЕНА")
    print("="*60)

    # Формируем URL для получения токена
    auth_url = f"https://oauth.yandex.ru/authorize?response_type=token&client_id={CLIENT_ID}"

    print("\nОткрываем браузер для авторизации...")
    print(f"Ссылка: {auth_url}")

    # Открываем браузер автоматически
    webbrowser.open(auth_url)

    print("\nВ браузере:")
    print("   - Авторизуйтесь в Яндексе")
    print("   - Разрешите доступ приложению")
    print("   - После авторизации вы попадёте на страницу с токеном в адресной строке")
    print("\nСкопируйте токен из адресной строки браузера")
    print("   (часть после #access_token= и до &token_type)")
    print("-"*60)

    # Получаем токен от пользователя
    token = input("\n Вставьте токен сюда: ").strip()

    # Простая валидация токена
    if not token or len(token) < 10:
        print("Неверный токен!")
        return None

    # Сохраняем токен
    with open(TOKEN_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'access_token': token,
            'client_id': CLIENT_ID,
            'created_at': str(__import__('datetime').datetime.now())
        }, f, ensure_ascii=False, indent=2)

    print(f"\nТокен сохранён в файл {TOKEN_FILE}")
    return token

def load_token():
    """Загружает сохранённый токен"""
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            token = data.get('access_token')
            if token:
                print(f"Загружен токен из {TOKEN_FILE}")
                return token
    return None

def ensure_token():
    """Гарантирует наличие токена (загружает или получает новый)"""
    token = load_token()
    if not token:
        token = get_token_automated()
        if not token:
            raise Exception("Не удалось получить токен")
    return token

def get_headers(access_token):
    """Возвращает заголовки для запросов"""
    return {
        "Authorization": f"OAuth {access_token}",
        "Accept": "application/json",
        "Content-Type": "application/json"
    }

def test_connection(access_token):
    """Проверяет работоспособность токена"""
    url = f"{API_BASE}/"
    response = requests.get(url, headers=get_headers(access_token))

    if response.status_code == 200:
        print("Соединение с Яндекс.Диском установлено!")
        return True
    else:
        print(f"Ошибка соединения: {response.status_code}")
        if response.status_code == 401:
            print("   Токен недействителен. Удалите файл yadisk_token.json и запустите скрипт заново.")
        return False

def upload_file(local_path, remote_path, access_token):
    """Загружает файл на Яндекс.Диск"""
    print(f"\nЗагрузка файла: {local_path} -> {remote_path}")

    # 1. Получаем ссылку для загрузки
    url = f"{API_BASE}/resources/upload"
    params = {"path": remote_path, "overwrite": "true"}
    response = requests.get(url, headers=get_headers(access_token), params=params)

    if response.status_code != 200:
        print(f"Ошибка получения ссылки: {response.status_code}")
        print(f"Ответ: {response.text}")
        return False

    upload_url = response.json().get("href")

    # 2. Загружаем файл
    try:
        with open(local_path, 'rb') as f:
            upload_response = requests.put(upload_url, files={"file": f})

        if upload_response.status_code == 201:
            print(f"Файл успешно загружен!")
            return True
        else:
            print(f"Ошибка загрузки: {upload_response.status_code}")
            return False
    except Exception as e:
        print(f"Исключение при загрузке: {e}")
        return False

def list_folder(folder_path, access_token):
    """Показывает содержимое папки"""
    url = f"{API_BASE}/resources"
    params = {"path": folder_path, "limit": 100}
    response = requests.get(url, headers=get_headers(access_token), params=params)

    if response.status_code == 200:
        items = response.json().get("_embedded", {}).get("items", [])
        print(f"\nСодержимое папки {folder_path}:")
        if not items:
            print("   Папка пуста")
        for item in items:
            print(f"  - {item.get('name')} ({item.get('type')}, {item.get('size', 0)} байт)")
        return True
    elif response.status_code == 404:
        print(f"Папка {folder_path} не существует")
        return False
    else:
        print(f"Ошибка: {response.status_code}")
        return False

def create_folder(folder_path, access_token):
    """Создаёт папку на Яндекс.Диске"""
    url = f"{API_BASE}/resources"
    params = {"path": folder_path}
    response = requests.put(url, headers=get_headers(access_token), params=params)

    if response.status_code == 201:
        print(f"Папка создана: {folder_path}")
        return True
    elif response.status_code == 409:
        print(f"Папка уже существует: {folder_path}")
        return True
    else:
        print(f"Ошибка создания папки: {response.status_code}")
        return False

def upload_folder(local_folder, remote_folder, access_token):
    """Загружает всю папку рекурсивно"""
    print(f"\nЗагрузка папки: {local_folder} -> {remote_folder}")

    if not os.path.exists(local_folder):
        print(f"Папка не найдена: {local_folder}")
        return False

    # Создаём корневую папку
    create_folder(remote_folder, access_token)

    success_count = 0
    error_count = 0

    for root, dirs, files in os.walk(local_folder):
        # Вычисляем относительный путь
        rel_path = os.path.relpath(root, local_folder)
        if rel_path == '.':
            current_remote = remote_folder
        else:
            current_remote = f"{remote_folder}/{rel_path}".replace('\\', '/')
            create_folder(current_remote, access_token)

        # Загружаем файлы
        for file in files:
            local_file = os.path.join(root, file)
            remote_file = f"{current_remote}/{file}".replace('\\', '/')

            if upload_file(local_file, remote_file, access_token):
                success_count += 1
            else:
                error_count += 1

    print(f"\nИтог: загружено {success_count} файлов, ошибок {error_count}")
    return error_count == 0

def main():
    print("\n" + "="*60)
    print("АВТОМАТИЗИРОВАННЫЙ КЛИЕНТ ЯНДЕКС.ДИСКА")
    print("="*60 + "\n")

    # 1. Получаем токен (автоматически загружаем или запрашиваем)
    try:
        access_token = ensure_token()
    except Exception as e:
        print(f"Ошибка: {e}")
        return

    # 2. Проверяем соединение
    if not test_connection(access_token):
        # Если токен не работает, удаляем его и пробуем снова
        if os.path.exists(TOKEN_FILE):
            os.remove(TOKEN_FILE)
            print("Удалён старый токен. Попробуйте запустить скрипт заново.")
        return

    # 3. Меню действий
    while True:
        print("\n" + "-"*60)
        print("ДОСТУПНЫЕ ДЕЙСТВИЯ:")
        print("  1. Загрузить файл")
        print("  2. Загрузить папку")
        print("  3. Показать содержимое папки")
        print("  4. Создать папку")
        print("  0. Выйти")
        print("-"*60)

        choice = input("Выберите действие (0-4): ").strip()

        if choice == '0':
            print("До свидания!")
            break

        elif choice == '1':
            local = input("Введите путь к локальному файлу: ").strip()
            remote = input("Введите путь на Яндекс.Диске (например, /ER/file.txt): ").strip()
            if local and remote:
                upload_file(local, remote, access_token)
            else:
                print("Пути не могут быть пустыми")

        elif choice == '2':
            local = input("Введите путь к локальной папке: ").strip()
            remote = input("Введите путь на Яндекс.Диске (например, /ER/backup): ").strip()
            if local and remote:
                upload_folder(local, remote, access_token)
            else:
                print("Пути не могут быть пустыми")

        elif choice == '3':
            remote = input("Введите путь на Яндекс.Диске (например, /ER): ").strip()
            if remote:
                list_folder(remote, access_token)
            else:
                print("Путь не может быть пустым")

        elif choice == '4':
            remote = input("Введите путь для новой папки (например, /ER/new_folder): ").strip()
            if remote:
                create_folder(remote, access_token)
            else:
                print("Путь не может быть пустым")

        else:
            print("Неверный выбор")

if __name__ == "__main__":
    main()


АВТОМАТИЗИРОВАННЫЙ КЛИЕНТ ЯНДЕКС.ДИСКА

Загружен токен из yadisk_token.json
Соединение с Яндекс.Диском установлено!

------------------------------------------------------------
ДОСТУПНЫЕ ДЕЙСТВИЯ:
  1. Загрузить файл
  2. Загрузить папку
  3. Показать содержимое папки
  4. Создать папку
  0. Выйти
------------------------------------------------------------
Выберите действие (0-4): 1
Введите путь к локальному файлу: /content/room-27.json.enc
Введите путь на Яндекс.Диске (например, /ER/file.txt): /ER/room-27.json.enc

Загрузка файла: /content/room-27.json.enc -> /ER/room-27.json.enc
Файл успешно загружен!

------------------------------------------------------------
ДОСТУПНЫЕ ДЕЙСТВИЯ:
  1. Загрузить файл
  2. Загрузить папку
  3. Показать содержимое папки
  4. Создать папку
  0. Выйти
------------------------------------------------------------
Выберите действие (0-4): 1
Введите путь к локальному файлу: /content/room-28.json.enc
Введите путь на Яндекс.Диске (например, /ER/file.txt): 